# Urban Step 4: Export TensorRT Engine Model (`urban_conditioned_lane_model.engine`)

This notebook compiles your trained **Urban Conditioned Lane Model** (`urban_conditioned_lane_model.onnx`) directly into an **NVIDIA TensorRT Engine file (`urban_conditioned_lane_model.engine`)** with **FP16 precision** for maximum GPU acceleration on Jetson Nano.

### 1. Setup Environment & Locate ONNX Model File

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

onnx_path = os.path.join(Path.cwd(), "urban_conditioned_lane_model.onnx")
engine_path = os.path.join(Path.cwd(), "urban_conditioned_lane_model.engine")

if not os.path.exists(onnx_path):
    print(f"[!] ERROR: ONNX model file '{onnx_path}' not found! Run urban_03_train_conditioned_lane_model.ipynb first.")
else:
    print(f"[+] Found ONNX model: {onnx_path}")
    print(f"[*] Target TensorRT Engine output: {engine_path}")


### 2. Compile ONNX Model to TensorRT Engine (`trtexec` FP16 Mode)

In [ ]:
trtexec_bin = "/usr/src/tensorrt/bin/trtexec"
if not os.path.exists(trtexec_bin):
    trtexec_bin = "trtexec"

print(f"[*] Starting TensorRT FP16 Compilation with {trtexec_bin}...")
cmd = [
    trtexec_bin,
    f"--onnx={onnx_path}",
    f"--saveEngine={engine_path}",
    "--fp16",
    "--workspace=1024"
]

print(f"[*] Running command: {' '.join(cmd)}\n")
try:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    
    if process.returncode == 0 and os.path.exists(engine_path):
        print(f"\n[+] SUCCESS! Urban TensorRT Engine compiled & saved -> '{engine_path}'")
    else:
        print(f"\n[!] trtexec returned exit code: {process.returncode}")
except Exception as e:
    print(f"\n[!] trtexec notice: {e}")
